# Lab 3 Task 4: Multirate Processing of a Real N310 Capture

Generate the tone file, capture it with the same finite GRC flowgraphs, predict each alias, and compare raw downsampling with filtered resampling.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

def locate_lab_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "support" / "lab3_tools.py").is_file():
            return base
    raise FileNotFoundError("Open this notebook from its Lab 3 folder.")

LAB_ROOT = locate_lab_root()
sys.path.insert(0, str(LAB_ROOT / "support"))
WORK_DIR = LAB_ROOT / "work"
FIGURE_DIR = WORK_DIR / "figures"
WAVEFORM_DIR = WORK_DIR / "waveforms"
CAPTURE_DIR = WORK_DIR / "captures"
for path in (FIGURE_DIR, WAVEFORM_DIR, CAPTURE_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Lab root:", LAB_ROOT)

Lab root: /home/sneill/ECE7377


In [2]:
from lab3_dsp import principal_alias
from lab3_tools import analyze_tone_capture, generate_tone_file, plot_task4_analysis

sample_rate_sps = 1_250_000
tone_hz = 100_000
tone_duration_s = 0.25
tone_amplitude = 0.10
transmit_seconds = 20.0
capture_seconds = 8.0

# to do: derive the output rates and predicted aliases.
rate_factor2_sps = sample_rate_sps // 2
rate_ratio7_10_sps = sample_rate_sps * 7 // 10
rate_factor20_sps = sample_rate_sps // 20
alias_factor2_hz = principal_alias(tone_hz, rate_factor2_sps)
alias_ratio7_10_hz = principal_alias(tone_hz, rate_ratio7_10_sps)
alias_factor20_hz = principal_alias(tone_hz, rate_factor20_sps)
tone_file_samples = int(sample_rate_sps * tone_duration_s)
tone_file_bytes = tone_file_samples * 8
capture_samples = int(sample_rate_sps * capture_seconds)
capture_bytes = capture_samples * 8

In [3]:
assert rate_factor2_sps == 625_000
assert rate_ratio7_10_sps == 875_000
assert rate_factor20_sps == 62_500
assert alias_factor2_hz == 100_000
assert alias_ratio7_10_hz == 100_000
assert alias_factor20_hz == -25_000
assert tone_file_samples == 312_500
assert tone_file_bytes == 2_500_000
assert capture_samples == 10_000_000
assert capture_bytes == 80_000_000
print("Rate, alias, and file-size calculations passed.")

Rate, alias, and file-size calculations passed.


In [4]:
tone_path = WAVEFORM_DIR / "task4_tone_tx_c64.dat"
tone = generate_tone_file(
    tone_path, sample_rate_sps, tone_hz, tone_duration_s, tone_amplitude
)
print(tone_path, tone_path.stat().st_size, "bytes", "peak", np.max(np.abs(tone)))

/home/sneill/ECE7377/work/waveforms/task4_tone_tx_c64.dat 2500000 bytes peak 0.10000001


## GNU Radio capture sequence

Use the same two GRC files as Task 3. In the transmitter, select `task4_tone_tx_c64.dat`, use a 20.0-second Head, and keep 2.45 GHz and 1.25 MS/s. In the receiver, select `task4_tone_c64.dat` and use an 8.0-second Head. Start the transmitter first, then start the receiver. Confirm 10,000,000 complex64 samples and 80,000,000 bytes.

In [ ]:
capture_path = CAPTURE_DIR / "task4_tone_c64.dat"
if not capture_path.is_file():
    raise FileNotFoundError("Capture task4_tone_c64.dat with GNU Radio before analysis.")
result = analyze_tone_capture(capture_path, sample_rate_sps, tone_hz)
for name, peak in result["peaks_hz"].items():
    print(f"{name:36s} peak {peak/1e3:9.3f} kHz, power {result['powers_db'][name]:8.3f} dB")
spectra_figure, alias_figure = plot_task4_analysis(result, FIGURE_DIR)
display(Image(filename=str(spectra_figure)))
display(Image(filename=str(alias_figure)))

<span style="color:#165696"><b>Report.</b> Include the tone file calculation, GRC settings, exact capture size, four spectra, predicted and measured peaks, and the raw-versus-filtered factor-20 comparison. Explain why one path aliases and the other suppresses the tone.</span>